In [1]:
pip install flwr torch numpy

Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [5]:
import pandas as pd
client_df = pd.read_csv("C:/Users/saipr/OneDrive/Documents/Deep Learning/FL-IDS Project/client2_data.csv")

In [7]:
from sklearn.model_selection import train_test_split

def split_client_data(client_df, test_size=0.2, val_size=0.2):
    X = client_df.drop(columns=["Label"])
    y = client_df["Label"]

    X_temp, X_test, y_temp, y_test = train_test_split(
        X, y, test_size=test_size, stratify=y, random_state=42
    )

    val_ratio_adjusted = val_size / (1 - test_size)
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=val_ratio_adjusted, stratify=y_temp, random_state=42
    )

    return X_train, X_val, X_test, y_train, y_val, y_test

X_train, X_val, X_test, y_train, y_val, y_test = split_client_data(client_df)

In [9]:
from model import LSTMIDS


In [11]:
import torch
from torch.utils.data import TensorDataset, DataLoader

def get_tensor_loaders(X_train, y_train, X_val, y_val, X_test, y_test, batch_size=32):
    train_dataset = TensorDataset(torch.tensor(X_train.values, dtype=torch.float32),
                                   torch.tensor(y_train.values, dtype=torch.float32))
    
    val_dataset = TensorDataset(torch.tensor(X_val.values, dtype=torch.float32),
                                 torch.tensor(y_val.values, dtype=torch.float32))

    test_dataset = TensorDataset(torch.tensor(X_test.values, dtype=torch.float32),
                                  torch.tensor(y_test.values, dtype=torch.float32))

    return DataLoader(train_dataset, batch_size=batch_size, shuffle=True), \
           DataLoader(val_dataset, batch_size=batch_size), \
           DataLoader(test_dataset, batch_size=batch_size)

train_loader, val_loader, test_loader = get_tensor_loaders(
    X_train, y_train, X_val, y_val, X_test, y_test
)

In [13]:
import flwr as fl
import torch
from torch import nn
from sklearn.metrics import precision_score, recall_score, f1_score
import csv
import os

# Do NOT initialize CrypTen in the client — server handles SMPC!

class IDSClient(fl.client.NumPyClient):
    def __init__(self, model, train_loader, val_loader):
        self.model = model
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.loss_fn = nn.BCELoss()
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=0.001)

    def get_parameters(self, config=None):
        return [val.cpu().detach().numpy() for val in self.model.parameters()]

    def set_parameters(self, parameters):
        for param, new_val in zip(self.model.parameters(), parameters):
            param.data = torch.tensor(new_val, dtype=param.data.dtype)

    def fit(self, parameters, config):
        self.set_parameters(parameters)
        self.model.train()
        for x_batch, y_batch in self.train_loader: 
            if len(x_batch.shape) == 2:
                x_batch = x_batch.unsqueeze(1)
            self.optimizer.zero_grad()
            y_pred = self.model(x_batch).squeeze()
            loss = self.loss_fn(y_pred, y_batch)
            loss.backward()
            self.optimizer.step()

        # Return plaintext parameters (SMPC happens on the server)
        updated_params = self.get_parameters()
        return updated_params, len(self.train_loader.dataset), {}

    def log_metrics(self, accuracy, precision, recall, f1, round_num):
        filename = f"client1_metrics_rounds.csv"
        file_exists = os.path.isfile(filename)

        with open(filename, mode='a', newline='') as file:
            writer = csv.writer(file)
            if not file_exists:
                writer.writerow(["Round", "Accuracy", "Precision", "Recall", "F1"])
            writer.writerow([round_num, accuracy, precision, recall, f1])

    def evaluate(self, parameters, config):
        self.set_parameters(parameters)
        self.model.eval()
        loss, correct, total = 0.0, 0, 0
        all_preds, all_labels = [], []

        with torch.no_grad():
            for x_batch, y_batch in self.val_loader:
                if len(x_batch.shape) == 2:
                    x_batch = x_batch.unsqueeze(1)
                y_pred = self.model(x_batch).squeeze()
                loss += self.loss_fn(y_pred, y_batch).item()

                preds = (y_pred >= 0.5).float()
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(y_batch.cpu().numpy())

                correct += (preds == y_batch).sum().item()
                total += y_batch.size(0)

        accuracy = correct / total
        precision = precision_score(all_labels, all_preds, zero_division=0)
        recall = recall_score(all_labels, all_preds, zero_division=0)
        f1 = f1_score(all_labels, all_preds, zero_division=0)

        print(f"Client1 Metrics: Accuracy={accuracy:.4f}, Precision={precision:.4f}, Recall={recall:.4f}, F1={f1:.4f}")

        round_num = int(config.get("round", -1))
        self.log_metrics(accuracy, precision, recall, f1, round_num)

        return float(loss), total, {
            "accuracy": float(accuracy),
            "precision": float(precision),
            "recall": float(recall),
            "f1_score": float(f1)
        }

# Create model and start client
from model import LSTMIDS  # Your shared model definition

model = LSTMIDS(input_size=25)
client = IDSClient(model, train_loader, val_loader)

fl.client.start_client(
    server_address="192.168.1.80:8080",
    client=client.to_client()
)

C:\Users\saipr\anaconda3\envs\crypten_env\lib\site-packages\torch\nn\modules\rnn.py:62: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn("dropout option adds dropout after all but last "
INFO :      
INFO :      Received: train message c1decd44-29ca-4a6d-b370-c4f4542a1cc4
INFO :      Sent reply
INFO :      
INFO :      Received: evaluate message 63cf6b62-b5a4-4ef0-9773-b953f5df865f
INFO :      Sent reply
INFO :      
INFO :      Received: train message 0e37be9b-8ff5-4da7-bf07-a07b0fce85f5


Client1 Metrics: Accuracy=0.9740, Precision=0.9490, Recall=0.9777, F1=0.9631


INFO :      Sent reply
INFO :      
INFO :      Received: evaluate message ea48ebb7-534f-49cc-a9c4-f9cad51789a0
INFO :      Sent reply
INFO :      
INFO :      Received: train message 49d15eb0-43e2-400b-b0ca-a40815075073


Client1 Metrics: Accuracy=0.9786, Precision=0.9598, Recall=0.9792, F1=0.9694


INFO :      Sent reply
INFO :      
INFO :      Received: evaluate message 2b2791ea-97ab-47a2-8451-1c6e08313334
INFO :      Sent reply
INFO :      
INFO :      Received: train message 4a099638-a00a-43fe-9ecc-4e100c33cff8


...

INFO :      Sent reply
INFO :      
INFO :      Received: evaluate message 4df2525f-581e-4e31-8867-ffd61020a1ad
INFO :      Sent reply
INFO :      
INFO :      Received: train message 476e663f-a241-43a9-a1c3-fec5627e34fc


Client1 Metrics: Accuracy=0.9911, Precision=0.9874, Recall=0.9871, F1=0.9872


INFO :      Sent reply
INFO :      
INFO :      Received: evaluate message 40b4d1b6-896a-4b9e-88ed-200618cca909
INFO :      Sent reply
INFO :      
INFO :      Received: reconnect message a6b58d0e-abda-4c40-97c1-da64af434d49


Client1 Metrics: Accuracy=0.9911, Precision=0.9862, Recall=0.9883, F1=0.9872


INFO :      Disconnect and shut down


In [ ]:
from model import LSTMIDS  # Already created

model = LSTMIDS(input_size=25)
client = IDSClient(model, train_loader, val_loader)

fl.client.start_client(
    server_address="192.168.1.80:8080",  # Replace with your server's IP
    client=client.to_client()
)